# 訓練 DeBERTa-v3-base（里程碑 2 + 里程碑 3 的 5-fold 驗證）

這是**訓練** notebook，跟推論 notebook（`infer_deberta.py`）分開 —— 訓練需要連網路
從 HuggingFace Hub 下載 base model 權重，但這題是 code competition，正式推論時
Kaggle 會**關閉網路**，所以離線推論必須另開一個 notebook，把這裡存出的權重當
Kaggle Dataset 掛進去用。

用 `kaggle kernels push` 上傳時（見 notebooks/README.md），kernel-metadata.json
已經設好 GPU、Internet On、`competition_sources` 掛這個競賽的資料 —— 不需要手動
在網頁上調整。手動在 Kaggle 網頁貼 cell 的話才需要自己設：
- Accelerator：GPU T4 x2（或 P100）
- Internet：On
- Add Data：這個競賽的資料集 `llm-classification-finetuning`

下面第一個 code cell 要換成 `notebooks/_bootstrap_cell.py` 的完整內容（用
`python scripts/gen_notebook_bootstrap.py > notebooks/_bootstrap_cell.py` 產生，
`src/llmcls/` 有改動就要重新產生一次再貼）。它會把 `src/llmcls/*.py` 的原始碼直接
寫進 `/kaggle/working/llmcls_src/`，不需要另外建 Kaggle Dataset 掛程式碼
——掛 Dataset 那條路容易在「有沒有建對 / 掛載名稱對不對」上出錯（`ModuleNotFoundError`）。

In [ ]:
"""貼進 Kaggle Notebook 第一個 cell —— 由 scripts/gen_notebook_bootstrap.py 產生，不要手改。"""

import pathlib
import sys

_llmcls_files = {
    '__init__.py': r'''"""LLM Classification Finetuning — 共用工具模組。

刻意寫成可 import 的模組（而不是單一 notebook），因為訓練會在
Kaggle Notebook 或遠端 GPU 上跑，本機只負責資料處理、CV 切分與提交組裝。
同一份程式碼兩邊都能 import，路徑差異由 config.py 吸收。
"""

from llmcls.config import DATA_DIR, LABEL_COLS, OUTPUT_DIR
from llmcls.metrics import UNIFORM_LOGLOSS, log_loss

__all__ = ["DATA_DIR", "OUTPUT_DIR", "LABEL_COLS", "log_loss", "UNIFORM_LOGLOSS"]
''',
    'calibration.py': r'''"""事後校準：temperature scaling。

log loss 吃的是機率校準品質，不是準確率——不用重新訓練，只要在驗證集的 logits 上
配一個純量溫度 T，讓 softmax(logits / T) 更校準，套用到測試集的 logits 上即可。

純 numpy，不需要 torch，可以在本機測試（真正的 logits 由 llmcls/train.py 的
predict_logits() / predict_logits_with_model() 產生，那兩個需要 torch/transformers）。
"""

from __future__ import annotations

import warnings

import numpy as np

from llmcls.metrics import log_loss, softmax


def apply_temperature(logits: np.ndarray, temperature: float) -> np.ndarray:
    """回傳 softmax(logits / temperature)。temperature > 1 會讓機率分佈變平滑
    （撫平過度自信），temperature < 1 會讓分佈更尖銳，temperature == 1 等於沒校準。
    """
    return softmax(np.asarray(logits) / temperature)


def fit_temperature(
    logits: np.ndarray,
    labels: np.ndarray,
    lo: float = 0.1,
    hi: float = 5.0,
    n_grid: int = 50,
    n_refine: int = 4,
) -> float:
    """在 [lo, hi] 網格搜尋、逐步細化，找出讓 log loss 最小的溫度 T。

    T 只有一個純量，網格搜尋 + 逐步細化就足夠穩定，不需要另外拉 scipy 依賴
    （sklearn 有牽帶到 scipy，但那是 transitive dependency，不想仰賴它）。
    """
    logits = np.asarray(logits)
    labels = np.asarray(labels)
    orig_lo, orig_hi = lo, hi
    best_t = 1.0
    cur_lo, cur_hi = lo, hi
    for _ in range(n_refine):
        candidates = np.linspace(cur_lo, cur_hi, n_grid)
        losses = [log_loss(labels, apply_temperature(logits, t)) for t in candidates]
        idx = int(np.argmin(losses))
        best_t = float(candidates[idx])
        span = (cur_hi - cur_lo) / n_grid
        cur_lo, cur_hi = max(1e-3, best_t - span), best_t + span

    if best_t <= orig_lo * 1.05 or best_t >= orig_hi * 0.95:
        warnings.warn(
            f"fit_temperature 找到的 T={best_t:.3f} 貼著搜尋邊界 [{orig_lo}, {orig_hi}]，"
            "可能沒收斂，先擴大 lo/hi 範圍再看一次"
        )
    return best_t
''',
    'config.py': r'''"""路徑與常數。本機 / Kaggle Notebook 的差異全部集中在這裡。"""

from __future__ import annotations

import os
from pathlib import Path

COMPETITION = "llm-classification-finetuning"

# Kaggle Notebook 內資料掛載路徑：透過網頁 UI「Add Data」掛的話是
# /kaggle/input/<competition>/，但實測透過 `kaggle kernels push`（kernel-metadata.json
# 的 competition_sources）掛的話，實際掛在 /kaggle/input/competitions/<competition>/，
# 多一層 competitions/ —— 兩條路徑都要認，不要假設只有一種。
# 可用環境變數 LLMCLS_DATA_DIR 覆寫（例如指到 data/fixture 跑煙霧測試）。
_KAGGLE_INPUT_CANDIDATES = [
    Path("/kaggle/input") / COMPETITION,
    Path("/kaggle/input/competitions") / COMPETITION,
]
# 本競賽目錄 competitions/<slug>/，不是 git repo 根目錄 —— 工作區還有其他競賽。
_COMP_ROOT = Path(__file__).resolve().parents[2]


def _resolve_data_dir() -> Path:
    if env := os.environ.get("LLMCLS_DATA_DIR"):
        return Path(env)
    for candidate in _KAGGLE_INPUT_CANDIDATES:
        if candidate.exists():
            return candidate
    return _COMP_ROOT / "data"


DATA_DIR = _resolve_data_dir()
# Kaggle Notebook 只有 /kaggle/working 可寫。
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else _COMP_ROOT / "outputs"

# 訓練與推論是兩個獨立的 Kaggle Notebook：訓練 notebook 把權重存到 OUTPUT_DIR/model，
# Save Version 後那個資料夾變成一個 Kaggle Dataset，掛進推論 notebook 時的掛載路徑
# 由使用者在 Kaggle UI 上決定、無法預先得知，所以用環境變數覆寫（呼應 LLMCLS_DATA_DIR）。
MODEL_DIR = Path(os.environ.get("LLMCLS_MODEL_DIR", str(OUTPUT_DIR / "model")))

# 訓練用的 base model；推論 notebook 離線，權重從 MODEL_DIR 讀，不會連到這個 hub id。
MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LEN = 512

TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_CSV = DATA_DIR / "sample_submission.csv"

# 三分類的目標欄位，順序即 class 0/1/2，全專案共用這個順序。
LABEL_COLS = ["winner_model_a", "winner_model_b", "winner_tie"]
N_CLASSES = len(LABEL_COLS)

# 需要 parse 的 JSON 字串欄位（多輪對話存成 list of str）。
TEXT_COLS = ["prompt", "response_a", "response_b"]

SEED = 42
N_FOLDS = 5
''',
    'cv.py': r'''"""交叉驗證切分。

重點：訓練集裡有數千筆重複的 prompt。如果同一個 prompt 同時落在 train 和 valid
fold，本地分數會虛高、跟 LB 對不上。所以一律以 prompt 當 group 切分。
"""

from __future__ import annotations

import hashlib

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

from llmcls.config import N_FOLDS, SEED


def prompt_group_key(df: pd.DataFrame) -> pd.Series:
    """以 prompt 文字的 hash 當 group id（正規化空白後再 hash）。"""
    normalized = df["prompt_text"].fillna("").str.strip().str.replace(r"\s+", " ", regex=True)
    return normalized.map(lambda s: hashlib.md5(s.encode("utf-8")).hexdigest())


def add_folds(
    df: pd.DataFrame, n_folds: int = N_FOLDS, seed: int = SEED, col: str = "fold"
) -> pd.DataFrame:
    """加上 fold 欄位：以 prompt 分組、以 label 分層。回傳新的 DataFrame。"""
    df = df.copy()
    groups = prompt_group_key(df)
    n_groups = groups.nunique()
    if n_groups < n_folds:
        raise ValueError(f"唯一 prompt 數 ({n_groups}) 少於 fold 數 ({n_folds})，無法切分")

    splitter = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    df[col] = -1
    for fold, (_, valid_idx) in enumerate(splitter.split(df, df["label"], groups)):
        df.iloc[valid_idx, df.columns.get_loc(col)] = fold

    assert (df[col] >= 0).all(), "有列沒有被分配到 fold"
    _assert_no_group_leak(df, groups, col)
    return df


def _assert_no_group_leak(df: pd.DataFrame, groups: pd.Series, col: str) -> None:
    """驗證每個 prompt group 只出現在單一 fold —— 這是切分的重點，值得直接斷言。"""
    per_group = pd.DataFrame({"group": groups.to_numpy(), "fold": df[col].to_numpy()})
    leaked = per_group.groupby("group")["fold"].nunique()
    n_leaked = int((leaked > 1).sum())
    if n_leaked:
        raise AssertionError(f"有 {n_leaked} 個 prompt group 跨越多個 fold，切分有誤")


def fold_indices(df: pd.DataFrame, fold: int, col: str = "fold") -> tuple[np.ndarray, np.ndarray]:
    """回傳 (train_idx, valid_idx) 的位置索引。"""
    is_valid = (df[col] == fold).to_numpy()
    return np.flatnonzero(~is_valid), np.flatnonzero(is_valid)
''',
    'data.py': r'''"""資料載入與欄位解析。

注意：prompt / response_a / response_b 在原始 CSV 裡是 **JSON 編碼的字串陣列**
（多輪對話，每個 element 是一輪），不是純文字。這點在真實資料下載前尚未實地驗證，
所以 parse 採防禦式寫法：json.loads 失敗就退回當成單輪純文字。
"""

from __future__ import annotations

import json
import warnings
from pathlib import Path

import pandas as pd

from llmcls.config import LABEL_COLS, TEST_CSV, TEXT_COLS, TRAIN_CSV

TURN_SEP = "\n\n"

# 退回純文字的比例超過這個門檻就直接報錯 —— 代表該欄根本不是 JSON，schema 假設錯了。
FALLBACK_ERROR_RATIO = 0.01


def _is_missing(raw: object) -> bool:
    """型別無關的缺值判斷（None / float nan / pd.NA 都算）。"""
    if raw is None:
        return True
    try:
        return bool(pd.isna(raw))
    except (TypeError, ValueError):
        # 例如 list、ndarray：pd.isna 回傳陣列或直接拋錯，都當作非缺值。
        return False


def _parse_turns_flagged(raw: object) -> tuple[list[str], bool]:
    """回傳 (turns, 是否退回純文字)。退回的次數會被上層統計，不能靜默吞掉。"""
    if _is_missing(raw):
        return [], False
    if isinstance(raw, list):
        return [("" if t is None else str(t)) for t in raw], False
    text = str(raw)
    try:
        parsed = json.loads(text)
    except (json.JSONDecodeError, ValueError):
        # 不是合法 JSON —— 當成單輪純文字，避免整批資料因為個別壞格式而中斷。
        return [text], True
    if isinstance(parsed, list):
        return [("" if t is None else str(t)) for t in parsed], False
    return [str(parsed)], True


def parse_turns(raw: object) -> list[str]:
    """把一格 JSON 字串解析成 list[str]；缺值 / 解析失敗都不會炸掉。"""
    return _parse_turns_flagged(raw)[0]


def join_turns(turns: list[str]) -> str:
    return TURN_SEP.join(t for t in turns if t)


def _require_columns(df: pd.DataFrame, cols: list[str], source: Path) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"{source} 缺少預期欄位 {missing}；實際欄位為 {list(df.columns)}。"
            " 若官方 schema 有變動，請同步更新 llmcls/config.py。"
        )


def _add_parsed_text(df: pd.DataFrame) -> pd.DataFrame:
    """解析文字欄位，並統計有多少列走了「退回純文字」的退路。

    這個計數是 schema 假設是否成立的第一手診斷：少數幾列是雜訊，比例一高就代表
    該欄根本不是 JSON。絕對不能靜默退回，否則模型會拿著原始 JSON 字串在訓練。
    """
    for col in TEXT_COLS:
        flagged = df[col].map(_parse_turns_flagged)
        turns = flagged.map(lambda x: x[0])
        n_fallback = int(flagged.map(lambda x: x[1]).sum())
        if n_fallback:
            msg = f"{col}：{n_fallback}/{len(df)} 列無法解析為 JSON 陣列，已退回單輪純文字"
            if n_fallback > FALLBACK_ERROR_RATIO * len(df):
                raise ValueError(
                    f"{msg} —— 比例過高，該欄的 schema 可能與預期不符。"
                    " 請檢查實際資料格式並更新 llmcls/data.py 的解析邏輯。"
                )
            warnings.warn(msg, stacklevel=2)
        df[f"{col}_turns"] = turns
        df[f"{col}_text"] = turns.map(join_turns)
    return df


def load_train(path: Path | None = None) -> pd.DataFrame:
    """載入訓練集，附上解析後的文字欄位與整數 label。"""
    path = Path(path) if path is not None else TRAIN_CSV
    if not path.exists():
        raise FileNotFoundError(f"找不到 {path}；請先執行 scripts/download_data.sh 下載競賽資料。")
    df = pd.read_csv(path)
    _require_columns(df, ["id", *TEXT_COLS, *LABEL_COLS], path)

    onehot = df[LABEL_COLS].to_numpy()
    bad = onehot.sum(axis=1) != 1
    if bad.any():
        raise ValueError(
            f"{path} 有 {int(bad.sum())} 列的 {LABEL_COLS} 不是恰好一個 1，"
            " 無法轉成單一 label，請檢查資料。"
        )
    df["label"] = onehot.argmax(axis=1)
    return _add_parsed_text(df)


def load_test(path: Path | None = None) -> pd.DataFrame:
    """載入測試集（沒有 label 欄）。"""
    path = Path(path) if path is not None else TEST_CSV
    if not path.exists():
        raise FileNotFoundError(f"找不到 {path}；請先執行 scripts/download_data.sh 下載競賽資料。")
    df = pd.read_csv(path)
    _require_columns(df, ["id", *TEXT_COLS], path)
    return _add_parsed_text(df)
''',
    'metrics.py': r'''"""評分指標。競賽用 multi-class log loss。"""

from __future__ import annotations

import numpy as np

from llmcls.config import N_CLASSES

# 均勻亂猜的分數 = ln(3) ≈ 1.0986。任何模型沒打敗這條線就等於沒有資訊量。
UNIFORM_LOGLOSS = float(np.log(N_CLASSES))

EPS = 1e-15


def softmax(x: np.ndarray) -> np.ndarray:
    x = x - x.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)


def log_loss(y_true: np.ndarray, y_prob: np.ndarray, eps: float = EPS) -> float:
    """multi-class log loss。y_true 是整數 label，y_prob 是 (n, n_classes) 機率。

    先 clip 到 [eps, 1-eps] 再逐列重新正規化。Kaggle 端的實際實作未經查證，
    但只要機率沒有極端到觸及 eps，各種變體的差異可忽略。
    """
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    if y_prob.ndim != 2 or y_prob.shape[1] != N_CLASSES:
        raise ValueError(f"y_prob 形狀應為 (n, {N_CLASSES})，實際為 {y_prob.shape}")
    if len(y_true) != len(y_prob):
        raise ValueError(f"y_true ({len(y_true)}) 與 y_prob ({len(y_prob)}) 長度不一致")
    if not np.isfinite(y_prob).all():
        raise ValueError("y_prob 含有 NaN 或 inf")

    p = np.clip(y_prob, eps, 1 - eps)
    p = p / p.sum(axis=1, keepdims=True)
    return float(-np.mean(np.log(p[np.arange(len(y_true)), y_true])))
''',
    'submission.py': r'''"""提交檔組裝與驗證。

Kaggle 只會告訴你「submission 格式錯誤」，不會告訴你錯在哪裡，
所以在本機就把能檢查的都檢查掉。
"""

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from llmcls.config import LABEL_COLS, N_CLASSES, OUTPUT_DIR, SAMPLE_SUBMISSION_CSV


def build_submission(ids: pd.Series | np.ndarray, probs: np.ndarray) -> pd.DataFrame:
    probs = np.asarray(probs, dtype=float)
    if probs.ndim != 2 or probs.shape[1] != N_CLASSES:
        raise ValueError(f"probs 形狀應為 (n, {N_CLASSES})，實際為 {probs.shape}")
    if len(ids) != len(probs):
        raise ValueError(f"ids ({len(ids)}) 與 probs ({len(probs)}) 長度不一致")
    sub = pd.DataFrame({"id": np.asarray(ids)})
    sub[LABEL_COLS] = probs
    return sub


def validate_submission(sub: pd.DataFrame, sample_path: Path | None = None) -> None:
    """檢查欄位、數值範圍、機率和；有 sample_submission 時再比對 id 集合。"""
    expected_cols = ["id", *LABEL_COLS]
    if list(sub.columns) != expected_cols:
        raise ValueError(f"欄位應為 {expected_cols}，實際為 {list(sub.columns)}")

    probs = sub[LABEL_COLS].to_numpy(dtype=float)
    if not np.isfinite(probs).all():
        raise ValueError("提交檔含有 NaN 或 inf")
    if (probs < 0).any() or (probs > 1).any():
        raise ValueError("機率值超出 [0, 1] 範圍")
    row_sums = probs.sum(axis=1)
    if not np.allclose(row_sums, 1.0, atol=1e-6):
        worst = float(np.abs(row_sums - 1.0).max())
        raise ValueError(f"每列機率和必須為 1，最大偏差 {worst:.2e}")
    if sub["id"].duplicated().any():
        raise ValueError("提交檔有重複的 id")

    sample_path = Path(sample_path) if sample_path is not None else SAMPLE_SUBMISSION_CSV
    if sample_path.exists():
        expected_ids = set(pd.read_csv(sample_path)["id"])
        actual_ids = set(sub["id"])
        if expected_ids != actual_ids:
            raise ValueError(
                f"id 集合與 sample_submission 不符："
                f"缺少 {len(expected_ids - actual_ids)} 筆、多出 {len(actual_ids - expected_ids)} 筆"
            )


def save_submission(
    sub: pd.DataFrame, name: str = "submission.csv", sample_path: Path | None = None
) -> Path:
    validate_submission(sub, sample_path)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    path = OUTPUT_DIR / name
    sub.to_csv(path, index=False)
    return path
''',
    'text.py': r'''"""把 prompt / response_a / response_b 的 token id 組成單一模型輸入序列，
超過 max_len 時做 head+tail 截斷。

刻意在 id 層級操作（呼叫端先分別 encode 三段，這裡只管截斷與長度預算），而不是
組字串再整段重新 tokenize —— 這樣三段可以各自截斷，不會因為 response_a 太長就把
response_b 擠到只剩尾巴一小段，兩邊被模型看到的資訊量比較公平。

不依賴 torch / transformers，純 Python 可在本機測試。真正呼叫 tokenizer 的地方在
llmcls/train.py（那裡才需要 GPU 環境）。
"""

from __future__ import annotations

# 對應 [CLS] prompt [SEP] response_a [SEP] response_b [SEP] 的組法：1 個 CLS + 3 個 SEP。
NUM_SPECIAL_TOKENS = 4

# 三段的預算比例：prompt 通常比兩份回覆短，兩份回覆同等重要所以各拿更多。
DEFAULT_RATIOS = (0.2, 0.4, 0.4)


def truncate_ids(ids: list[int], budget: int, head_ratio: float = 0.5) -> list[int]:
    """留頭尾、砍中間。budget <= 0 回傳空列表，不需要截斷時原樣回傳。"""
    if budget <= 0:
        return []
    if len(ids) <= budget:
        return ids
    head_len = min(budget, max(1, round(budget * head_ratio)))
    tail_len = budget - head_len
    if tail_len <= 0:
        return ids[:head_len]
    return ids[:head_len] + ids[len(ids) - tail_len :]


def split_budget(total: int, ratios: tuple[float, float, float] = DEFAULT_RATIOS) -> tuple[int, int, int]:
    """依 ratios 把 total 分給三段；四捨五入的誤差全部歸給最後一段，確保三段總和精確等於 total。"""
    if total <= 0:
        return (0, 0, 0)
    a = int(total * ratios[0])
    b = int(total * ratios[1])
    c = total - a - b
    return (a, b, c)


def build_input_ids(
    prompt_ids: list[int],
    response_a_ids: list[int],
    response_b_ids: list[int],
    max_len: int,
    ratios: tuple[float, float, float] = DEFAULT_RATIOS,
    head_ratio: float = 0.5,
) -> tuple[list[int], list[int], list[int]]:
    """回傳截斷後的 (prompt_ids, response_a_ids, response_b_ids)。

    呼叫端還要自己補上 CLS/SEP 特殊 token，所以保證
    len(p) + len(a) + len(b) + NUM_SPECIAL_TOKENS <= max_len。
    """
    budget = max_len - NUM_SPECIAL_TOKENS
    if budget <= 0:
        raise ValueError(f"max_len ({max_len}) 太小，容不下 {NUM_SPECIAL_TOKENS} 個特殊 token")
    p_budget, a_budget, b_budget = split_budget(budget, ratios)
    return (
        truncate_ids(prompt_ids, p_budget, head_ratio),
        truncate_ids(response_a_ids, a_budget, head_ratio),
        truncate_ids(response_b_ids, b_budget, head_ratio),
    )
''',
    'train.py': r'''"""DeBERTa-v3-base 三分類微調：資料集組裝、訓練、推論。

只能在有 torch / transformers 的環境 import（Kaggle Notebook 或有 GPU 的機器）；
本機沒有 GPU，這個檔案沒有、也無法有本機測試覆蓋。截斷邏輯本身在 llmcls/text.py
裡用純 Python 測試過，這裡只是把它接上真正的 tokenizer 和 HF Trainer。

`train_fold()` 是核心入口，同時給兩種呼叫方式用：
- CLI：scripts/train.py（適合遠端 GPU，例如 RunPod）
- Kaggle Notebook cell：`from llmcls.train import train_fold` 直接呼叫
"""

from __future__ import annotations

import inspect
import math
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

from llmcls.config import MAX_LEN, MODEL_DIR, MODEL_NAME, N_CLASSES, N_FOLDS, SEED
from llmcls.cv import add_folds, fold_indices
from llmcls.data import load_test, load_train
from llmcls.metrics import UNIFORM_LOGLOSS, log_loss, softmax
from llmcls.submission import build_submission, save_submission
from llmcls.text import build_input_ids
from llmcls.tta import average_swapped


class PreferenceDataset(torch.utils.data.Dataset):
    """把三欄文字 tokenize 成單一序列：[CLS] prompt [SEP] response_a [SEP] response_b [SEP]。"""

    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int, labels: np.ndarray | None):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.labels = labels
        # 預先 encode 三欄一次，__getitem__ 只做截斷 + 拼接，不用每個 epoch 重複 tokenize。
        # 整批呼叫 tokenizer(...)，不要逐列呼叫 tokenizer.encode()——fast tokenizer 的
        # 平行化是在批次呼叫內部做的（Rust 那邊自己開執行緒），逐列呼叫每次都要重新付一次
        # Python↔Rust FFI 的固定開銷，資料量上萬列時這筆開銷不能忽略，TTA 又是同一批文字
        # 重新 tokenize 兩次（正常順序 + 對調順序），批次呼叫能把這個成本壓下去。
        self.prompt_ids = tokenizer(df["prompt_text"].tolist(), add_special_tokens=False)["input_ids"]
        self.response_a_ids = tokenizer(df["response_a_text"].tolist(), add_special_tokens=False)["input_ids"]
        self.response_b_ids = tokenizer(df["response_b_text"].tolist(), add_special_tokens=False)["input_ids"]

    def __len__(self) -> int:
        return len(self.prompt_ids)

    def __getitem__(self, idx: int) -> dict:
        p, a, b = build_input_ids(
            self.prompt_ids[idx], self.response_a_ids[idx], self.response_b_ids[idx], self.max_len
        )
        cls_id, sep_id = self.tokenizer.cls_token_id, self.tokenizer.sep_token_id
        input_ids = [cls_id, *p, sep_id, *a, sep_id, *b, sep_id]
        item = {"input_ids": input_ids, "attention_mask": [1] * len(input_ids)}
        if self.labels is not None:
            item["labels"] = int(self.labels[idx])
        return item


class StopOnNonFiniteLoss(TrainerCallback):
    """DeBERTa-v3 在 fp32、peak LR 附近實測會突然發散：loss 衝高、grad_norm 變 NaN，
    之後每一步都是壞的，權重永久壞掉但 Trainer 完全不知道、還是把剩下的 epoch 跑完
    （實測浪費了 83 分鐘 GPU 時間裡的 70 分鐘）。這裡一偵測到就叫它停，把剩下的時間
    省下來，`triggered` 讓呼叫端知道這次訓練發散過、權重不可信。
    """

    def __init__(self) -> None:
        self.triggered = False

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and any(
            isinstance(v, (int, float)) and not math.isfinite(v) for k, v in logs.items() if k in ("loss", "grad_norm")
        ):
            control.should_training_stop = True
            self.triggered = True
        return control


# log_loss 不可能自然達到的高值，只用來確保「發散」在 metric_for_best_model 排序上
# 一定輸給任何健康的 checkpoint（見 _compute_metrics 的說明）。
_NONFINITE_SENTINEL = 99.0


def _compute_metrics(eval_pred) -> dict:
    """log_loss() 對非法機率值嚴格報錯（這是它的正確行為，見 metrics.py）；但這裡是
    Trainer 的 eval callback，一炸整個 run 就白跑、連權重都存不到，所以不能用 raise。

    早期版本把非有限值夾到均勻機率再算分——結果發散的 checkpoint 剛好算出
    log loss = ln(3)，而 greater_is_better=False 之下 ln(3) 比任何健康 checkpoint
    的分數都「小」，load_best_model_at_end 反而會選中發散的那個。改成回報一個大到
    不可能自然出現的哨兵值，讓發散的 checkpoint 在排序上必輸。
    """
    logits, labels = eval_pred
    probs = softmax(np.asarray(logits))
    finite = np.isfinite(probs).all(axis=1)
    n_nonfinite = int((~finite).sum())
    if n_nonfinite:
        return {"log_loss": _NONFINITE_SENTINEL, "vs_uniform": float("nan"), "n_nonfinite": n_nonfinite}
    score = log_loss(labels, probs)
    return {"log_loss": score, "vs_uniform": UNIFORM_LOGLOSS - score, "n_nonfinite": n_nonfinite}


def _training_args(
    output_dir: Path,
    epochs: int,
    batch_size: int,
    lr: float,
    lr_scheduler_type: str = "linear",
    max_steps: int | None = None,
    eval_steps: int | None = None,
    fp16: bool = True,
    label_smoothing: float = 0.0,
) -> TrainingArguments:
    # transformers 把 evaluation_strategy 改名成 eval_strategy 過；Kaggle Notebook 內建的
    # 版本不固定，用 inspect 挑對的參數名比硬編一個更穩。
    params = inspect.signature(TrainingArguments.__init__).parameters
    strategy_key = "eval_strategy" if "eval_strategy" in params else "evaluation_strategy"
    # 一律用 steps（不是 epoch）當 eval/save 的節奏，完整訓練也一樣 —— 實測 DeBERTa-v3
    # 在 fp32 訓練到一半會發散，只在 epoch 邊界存檔的話，發散前那個還健康的檢查點根本
    # 沒機會被存下來，load_best_model_at_end 也就沒有東西可挑。1500 是給完整訓練用的
    # 預設值：跟 eval_subset_rows 搭配（train_fold 會把訓練中途的評估換成子集），
    # 拉開頻率不會犧牲發散偵測 —— 那是每 50 步看 loss/grad_norm 的 StopOnNonFiniteLoss
    # callback 在管，跟這裡的 eval 節奏無關。
    default_eval_steps = max(1, max_steps // 2) if max_steps is not None else 1500
    steps = eval_steps or default_eval_steps
    kwargs = dict(
        output_dir=str(output_dir),
        **{strategy_key: "steps"},
        save_strategy="steps",
        eval_steps=steps,
        save_steps=steps,
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="log_loss",
        greater_is_better=False,
        learning_rate=lr,
        lr_scheduler_type=lr_scheduler_type,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        num_train_epochs=epochs,
        warmup_ratio=0.1,
        weight_decay=0.01,
        # 第一次踩到的坑是模型「裸」用 fp16（checkpoint 原本就存 fp16，from_pretrained
        # 沒有轉型），完全沒有 loss scaler 保護。train_fold() 現在會先強制 model.float()
        # 轉成真正的 fp32，這裡的 fp16=True 才是正規流程：autocast 動態轉型 + GradScaler
        # 做 loss scaling，梯度真的非有限值時 GradScaler 會跳過那一步而不是把權重弄壞，
        # StopOnNonFiniteLoss 則是最後一道防線。T4 有 fp16 tensor core，這樣才吃得到
        # 混合精度的加速。
        fp16=fp16,
        # Trainer 內建的 label smoothing：labels 還是整數類別（不用先轉成 one-hot），
        # HF 的 LabelSmoother 會在算 cross entropy 時自動把目標機率從 1.0 壓低、
        # 分一點出去給另外兩類。0.0 等於關閉，行為跟原本完全一樣。
        label_smoothing_factor=label_smoothing,
        report_to=[],
        logging_steps=10 if max_steps else 50,
        # 關掉 tqdm 進度條、強制用純文字 print 記錄 loss —— Kaggle Notebook 預設會用
        # rich/widget 進度條，那些輸出只進 __notebook__.ipynb 的 cell output，不會出現
        # 在 `kaggle kernels output` 抓得到的純文字 log，事後完全查不到 loss 曲線。
        disable_tqdm=True,
        seed=42,
    )
    if max_steps is not None:
        kwargs["max_steps"] = max_steps
    return TrainingArguments(**kwargs)


def predict_logits(trainer: Trainer, tokenizer, df: pd.DataFrame, max_len: int) -> np.ndarray:
    """對沒有 label 的 DataFrame（例如 test set）跑推論，回傳 (n, N_CLASSES) 的原始 logits
    （softmax 之前）——temperature scaling 要在這個尺度上配溫度，不能對已經 softmax
    過的機率配。
    """
    ds = PreferenceDataset(df, tokenizer, max_len, labels=None)
    return np.asarray(trainer.predict(ds).predictions)


def predict_probs(trainer: Trainer, tokenizer, df: pd.DataFrame, max_len: int) -> np.ndarray:
    """對沒有 label 的 DataFrame（例如 test set）跑推論，回傳 (n, N_CLASSES) 機率。"""
    return softmax(predict_logits(trainer, tokenizer, df, max_len))


def swap_ab(df: pd.DataFrame) -> pd.DataFrame:
    """回傳 response_a_text / response_b_text 對調後的複本，其餘欄位不變 ——
    PreferenceDataset 只讀這兩欄跟 prompt_text 建輸入，對調這兩欄就等於把
    response_a / response_b 的順序整個倒過來重新推論一次。
    """
    swapped = df.copy()
    swapped["response_a_text"] = df["response_b_text"].to_numpy()
    swapped["response_b_text"] = df["response_a_text"].to_numpy()
    return swapped


def train_fold(
    fold: int = 0,
    model_name: str = MODEL_NAME,
    n_folds: int = N_FOLDS,
    max_len: int = MAX_LEN,
    epochs: int = 2,
    batch_size: int = 8,
    lr: float = 2e-5,
    lr_scheduler_type: str = "linear",
    fp16: bool = True,
    output_dir: Path | None = None,
    max_train_rows: int | None = None,
    max_valid_rows: int | None = None,
    max_steps: int | None = None,
    eval_steps: int | None = None,
    eval_subset_rows: int | None = None,
    label_smoothing: float = 0.0,
) -> dict:
    """練一個 fold，存權重，回傳 {"score", "n_nonfinite", "diverged", "output_dir",
    "trainer", "tokenizer"}。

    預設只練 fold 0，不是全部 n_folds —— 先確認贏過 baseline_prior.py 印出的分數，
    再決定要不要花時間跑滿整個 CV。

    `max_train_rows` / `max_valid_rows` / `max_steps` / `eval_steps` 是煙霧測試用的：
    隨機抽一小撮資料、跑幾十步就評估一次，把「資料→tokenize→forward→eval→存檔」整條
    路徑在幾分鐘內走過一遍，而不是每次改動都要賭一整個 epoch（30-60 分鐘 GPU 時間）
    才知道炸不炸。跑煙霧測試時務必把 `lr_scheduler_type` 設成 "constant_with_warmup"
    ——用預設的 "linear" 配上 `max_steps` 很小的話，學習率暖身完就立刻開始衰減，
    根本沒有停留在 peak LR 的時間，測不出「訓練到 peak LR 附近才發散」這種問題
    （這正是本專案第一次煙霧測試沒抓到、完整訓練卻在 peak LR 附近整個發散的原因）。

    `eval_subset_rows` 是完整訓練用的加速選項：訓練中途的週期性評估只在這個子集上跑
    （原本每次評估都對完整驗證集跑一次，實測光是評估就佔掉總訓練時間近一半），最後
    收斂完仍然會對完整驗證集重新 `evaluate()` 一次，回傳的 `score` 保證是完整驗證集
    的分數，不會被子集的雜訊污染。

    `label_smoothing`（milestone 3）：0.0 是關閉，跟原本行為一樣；HF Trainer 內建
    支援，不用自己改 labels 或 loss function。

    `batch_size` 調大要非常小心：煙霧測試只能驗證穩定性（會不會發散），驗不出「完整
    資料集上的記憶體上限」——`max_train_rows` 抽樣的子集很難剛好抽到全是接近
    `max_len` 上限的最壞情況那幾批。實測 batch_size=16 在 3000 筆的煙霧測試上完全
    穩定，換成完整的 45746 筆卻在訓練中途 CUDA OOM（T4 記憶體只差 66MB）。batch_size
    調大之前，煙霧測試過關不代表完整資料集上安全。
    """
    # Kaggle 的 GPU kernel 預設給 T4 x2；HF Trainer 偵測到多張卡會自動包成
    # nn.DataParallel，這是已知會在 eval 階段的 predictions gather 上出怪問題的來源
    # （一個訊號：「gather along dimension 0 ... all input tensors were scalars」的
    # warning）。deberta-v3-base 在 batch_size 8 / max_len 512 下單張 T4 就跑得動，
    # 沒有 DP 帶來的好處，直接限制成單卡排除這個變因。用 setdefault 而不是強制覆蓋，
    # 呼叫端仍可自行指定 CUDA_VISIBLE_DEVICES。
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

    output_dir = Path(output_dir) if output_dir is not None else MODEL_DIR / f"fold{fold}"

    train = load_train()
    print(f"train: {len(train)} 列")
    train = add_folds(train, n_folds=n_folds)
    tr_idx, va_idx = fold_indices(train, fold)
    print(f"fold {fold}: train {len(tr_idx)} 列, valid {len(va_idx)} 列")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=N_CLASSES)
    # 實測 deberta-v3-base 在 HF Hub 上是用 fp16 存的，新版 transformers 的
    # from_pretrained 預設照抄 checkpoint 原本的 dtype，跟 TrainingArguments(fp16=False)
    # 完全無關 —— 結果是在跑「沒有 loss scaler 保護的裸 fp16 訓練」，梯度撐不了多久
    # 就溢位成 NaN，調低學習率只是延後發生、不是解法。強制轉 fp32 才是真正對應
    # fp16=False 的意圖。
    print(f"model 載入時的 dtype：{next(model.parameters()).dtype}")
    model = model.float()

    tr_df = train.iloc[tr_idx].reset_index(drop=True)
    va_df = train.iloc[va_idx].reset_index(drop=True)
    # 用隨機抽樣而不是頭幾列 —— 頭幾列在煙霧測試時永遠是同一批，測不到資料的多樣性。
    if max_train_rows is not None:
        tr_df = tr_df.sample(n=min(max_train_rows, len(tr_df)), random_state=SEED).reset_index(drop=True)
    if max_valid_rows is not None:
        va_df = va_df.sample(n=min(max_valid_rows, len(va_df)), random_state=SEED).reset_index(drop=True)
    tr_ds = PreferenceDataset(tr_df, tokenizer, max_len, tr_df["label"].to_numpy())
    va_ds = PreferenceDataset(va_df, tokenizer, max_len, va_df["label"].to_numpy())

    # 訓練中途的週期性評估用子集（快很多），最後才對完整驗證集重新算一次真正的分數。
    if eval_subset_rows is not None and eval_subset_rows < len(va_df):
        va_df_periodic = va_df.sample(n=eval_subset_rows, random_state=SEED).reset_index(drop=True)
        va_ds_periodic = PreferenceDataset(va_df_periodic, tokenizer, max_len, va_df_periodic["label"].to_numpy())
    else:
        va_ds_periodic = va_ds

    stop_callback = StopOnNonFiniteLoss()
    trainer = Trainer(
        model=model,
        args=_training_args(
            output_dir, epochs, batch_size, lr, lr_scheduler_type=lr_scheduler_type,
            max_steps=max_steps, eval_steps=eval_steps, fp16=fp16,
            label_smoothing=label_smoothing,
        ),
        train_dataset=tr_ds,
        eval_dataset=va_ds_periodic,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=_compute_metrics,
        callbacks=[stop_callback],
    )
    trainer.train()
    if stop_callback.triggered:
        print("偵測到 loss/grad_norm 變成 NaN，已提前停止訓練 —— 這次的權重不可信，不要拿去推論")

    # 存檔緊接在 train() 後面、explicit evaluate() 之前 —— load_best_model_at_end=True
    # 已經把最佳權重換回 trainer.model，這裡先存起來，後面的 evaluate() 就算出狀況
    # 也不會白跑一整個 epoch 的 GPU 時間卻什麼都沒留下。
    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))

    # TrainingArguments 的 output_dir 跟這裡的最終存檔目錄是同一個路徑，Trainer 自己
    # 的 save_steps 週期性存檔（含 optimizer/scheduler state，體積是模型本身的 2-3
    # 倍）還留在 output_dir/checkpoint-*/ 底下，跟上面剛存好的最終權重是同一份東西的
    # 重複備份。單一 fold 時這筆多餘的空間還在 Kaggle 磁碟額度內，5 個 fold 一次跑完
    # 疊起來就會把磁碟塞爆（實測踩到：5 folds 沒清、跑到一半磁碟就滿了）。權重已經
    # 存到 output_dir 頂層，這些子目錄可以直接刪掉。
    for checkpoint_dir in output_dir.glob("checkpoint-*"):
        shutil.rmtree(checkpoint_dir)
    print(f"模型已存到 {output_dir}（訓練中途的 checkpoint-* 已清除）")

    # 明確傳完整的 va_ds —— 訓練中途用的可能是子集，最終回報的分數必須是完整驗證集
    # 算出來的，不能被子集的雜訊污染。
    metrics = trainer.evaluate(eval_dataset=va_ds)
    score = metrics["eval_log_loss"]
    n_nonfinite = metrics.get("eval_n_nonfinite", 0)
    delta = UNIFORM_LOGLOSS - score
    # n_nonfinite > 0 代表分數是拿均勻機率湊出來的假象（例如全部 clamp 之後 delta 剛好
    # 等於 0，會被誤判成「打平基準」）——只要有非有限值，不管 delta 多少一律算沒過關。
    passed = n_nonfinite == 0 and not stop_callback.triggered and delta > 0
    print(f"\nvalid log loss   {score:.5f}")
    print(f"均勻亂猜基準      {UNIFORM_LOGLOSS:.5f}  (ln 3)")
    print(f"改善              {delta:+.5f}  {'✓ 優於基準' if passed else '✗ 未優於基準'}")
    if n_nonfinite:
        print(f"警告：{n_nonfinite}/{len(va_df)} 筆驗證預測是 NaN/inf，已夾到均勻機率計分 —— 分數不可信，先查訓練穩定性")

    return {
        "score": score,
        "n_nonfinite": n_nonfinite,
        "diverged": stop_callback.triggered,
        "output_dir": output_dir,
        "trainer": trainer,
        "tokenizer": tokenizer,
    }


def load_trained(output_dir: Path):
    """從已存的權重目錄載入 model + tokenizer（供推論 notebook 用，不需要 Trainer）。"""
    tokenizer = AutoTokenizer.from_pretrained(str(output_dir))
    model = AutoModelForSequenceClassification.from_pretrained(str(output_dir)).float()
    return model, tokenizer


def predict_logits_with_model(model, tokenizer, df: pd.DataFrame, max_len: int, batch_size: int = 32) -> np.ndarray:
    """離線推論 notebook 用：不需要 Trainer / TrainingArguments，直接跑 forward，
    回傳 softmax 之前的原始 logits（temperature scaling 要配在這個尺度上）。
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device).eval()
    ds = PreferenceDataset(df, tokenizer, max_len, labels=None)
    collator = DataCollatorWithPadding(tokenizer=tokenizer)
    all_logits = []
    with torch.no_grad():
        for start in range(0, len(ds), batch_size):
            batch = [ds[i] for i in range(start, min(start + batch_size, len(ds)))]
            inputs = collator(batch).to(device)
            logits = model(**inputs).logits.detach().cpu().numpy()
            all_logits.append(logits)
    return np.concatenate(all_logits, axis=0)


def predict_with_model(model, tokenizer, df: pd.DataFrame, max_len: int, batch_size: int = 32) -> np.ndarray:
    """離線推論 notebook 用：不需要 Trainer / TrainingArguments，直接跑 forward。"""
    return softmax(predict_logits_with_model(model, tokenizer, df, max_len, batch_size))


def predict_logits_with_tta(model, tokenizer, df: pd.DataFrame, max_len: int, batch_size: int = 32) -> np.ndarray:
    """a/b 對調 TTA，回傳原始順序與對調順序（已換回欄位對齊）logits 的平均。

    對付位置偏誤：模型可能學到偏好放在 A 或 B 位置本身，而不是回覆的品質。回傳
    logits（不是機率）是為了跟 temperature scaling 串接——要接著配溫度的話，必須
    先在 logits 尺度合併成一組，再對『合併後的 logits』配溫度，在機率層級平均、
    再對已經攤平過的機率配溫度會失真。單純只要 TTA、不接 calibration 的話，直接
    對這裡回傳的結果做 softmax 即可。
    """
    logits_orig = predict_logits_with_model(model, tokenizer, df, max_len, batch_size)
    logits_swapped = predict_logits_with_model(model, tokenizer, swap_ab(df), max_len, batch_size)
    return average_swapped(logits_orig, logits_swapped)


def predict_with_tta(model, tokenizer, df: pd.DataFrame, max_len: int, batch_size: int = 32) -> np.ndarray:
    """a/b 對調 TTA：原始順序跟對調順序各推論一次，換回欄位對齊後在機率層級平均。"""
    probs_orig = predict_with_model(model, tokenizer, df, max_len, batch_size)
    probs_swapped = predict_with_model(model, tokenizer, swap_ab(df), max_len, batch_size)
    return average_swapped(probs_orig, probs_swapped)


def predict_test_and_save(trainer_result: dict, name: str = "submission.csv") -> Path:
    """train_fold() 回傳值直接餵進來，對 test.csv 推論並寫出提交檔。"""
    test = load_test()
    probs = predict_probs(trainer_result["trainer"], trainer_result["tokenizer"], test, MAX_LEN)
    sub = build_submission(test["id"], probs)
    return save_submission(sub, name=name)
''',
    'tta.py': r'''"""a/b 對調 TTA（test-time augmentation）：對付位置偏誤。

模型可能學到偏好某個位置（A 或 B）本身，而不是純粹依回覆品質判斷。把 response_a /
response_b 對調後再推論一次，兩次結果換回原本的欄位對齊後平均，讓最終預測不受
位置影響。

純 numpy，不需要 torch，可以在本機測試（真正的兩次推論在 llmcls/train.py，
需要 torch/transformers）。
"""

from __future__ import annotations

import numpy as np

# LABEL_COLS 的欄位順序：[winner_model_a, winner_model_b, winner_tie]。
_SWAP_COLUMNS = [1, 0, 2]


def average_swapped(arr_orig: np.ndarray, arr_swapped: np.ndarray) -> np.ndarray:
    """把對調順序推論出來的結果換回原本的 a/b 欄位對齊，再跟原始順序的結果平均。

    `arr_swapped` 是把 response_a/response_b 對調後推論出來的，它的欄位順序是
    [P(目前放在 A 位置的贏), P(目前放在 B 位置的贏), P(tie)]——但「目前放在 A 位置」
    的其實是原本的 response_b，所以要先把前兩欄位換回來（`arr_swapped[:, [1, 0, 2]]`），
    才能跟 `arr_orig` 對齊平均。同一個操作對 logits（softmax 之前）跟機率（softmax
    之後）都適用，純粹是欄位重排 + 逐元素平均。
    """
    arr_orig = np.asarray(arr_orig)
    arr_swapped = np.asarray(arr_swapped)
    if arr_orig.shape != arr_swapped.shape:
        raise ValueError(f"形狀不一致：arr_orig {arr_orig.shape} vs arr_swapped {arr_swapped.shape}")
    aligned = arr_swapped[:, _SWAP_COLUMNS]
    return (arr_orig + aligned) / 2
''',
}

_pkg_dir = pathlib.Path("/kaggle/working/llmcls_src/src/llmcls")
_pkg_dir.mkdir(parents=True, exist_ok=True)
for _name, _content in _llmcls_files.items():
    (_pkg_dir / _name).write_text(_content, encoding="utf-8")

sys.path.insert(0, "/kaggle/working/llmcls_src/src")
print("llmcls bootstrapped:", sorted(p.name for p in _pkg_dir.glob("*.py")))

檢查一下 `/kaggle/input` 底下實際掛了什麼、`llmcls.config` 解析出來的路徑對不對 ——
`competition_sources` 掛的資料如果不在 `DATA_DIR` 猜的路徑，`load_train()` 會找不到
`train.csv`，這個 cell 就是拿來當場抓出真正的掛載路徑用的。

In [ ]:
import pathlib

print("/kaggle/input 底下（找 train.csv，最多往下 3 層）：")
for p in pathlib.Path("/kaggle/input").glob("**/train.csv"):
    print(" ", p)

from llmcls.config import DATA_DIR

print("llmcls.config.DATA_DIR =", DATA_DIR, " exists =", DATA_DIR.exists())
if DATA_DIR.exists():
    print("DATA_DIR 底下：", sorted(p.name for p in DATA_DIR.iterdir()))

若 Kaggle 內建的 transformers 版本太舊，才需要下面這行（通常不必要，內建已經夠新）。

In [ ]:
# !pip install -q -U transformers accelerate

**先跑煙霧測試，不要直接跑整個 epoch。** 里程碑 2 第一次完整訓練（fold 0、
batch_size=8、fp32、每 500 步對完整驗證集算一次分數）花了將近 9 小時，其中光是
評估就佔掉將近一半 —— 這裡在原本已經驗證過穩定的設定上加兩個加速手段，跑之前先用
煙霧測試確認沒有引入新的不穩定：

1. `fp16=True`：train_fold() 已經會先強制 `model.float()` 轉成真正的 fp32，
   這裡的 fp16 是正規的 autocast + GradScaler 混合精度 —— 跟里程碑 2 除錯過程中
   踩到的「裸 fp16、沒有 loss scaler」完全不同，梯度真的非有限值時 GradScaler
   會跳過那一步而不是把權重弄壞，`StopOnNonFiniteLoss` 仍是最後一道防線。
2. `eval_subset_rows`：訓練中途的週期性評估改成只對一小撮驗證集跑（快很多），
   最後才對完整驗證集重新算一次真正的分數 —— 發散偵測本來就跟評估頻率無關
   （`StopOnNonFiniteLoss` 每 50 步看一次 loss/grad_norm），拉開評估頻率不會
   犧牲安全網。

`batch_size` 原本也想從 8 調到 16 —— 煙霧測試（3000 筆子集）完全穩定通過，但換成
完整的 45746 筆資料後，訓練中途在某一批剛好全是接近 max_len 上限的長序列時 CUDA
OOM 了（T4 記憶體只差 66MB）。**煙霧測試只能驗證穩定性，驗不出完整資料集上的
記憶體上限** —— 子集抽樣很難剛好抽到最壞情況的那幾批，所以 `batch_size` 維持
里程碑 2 已經在完整資料集上證明過安全的 8，不冒這個風險。

煙霧測試維持跟里程碑 2 一樣的做法：`lr_scheduler_type="constant_with_warmup"`、
跑 400 步，讓學習率停在 peak 不再衰減，才測得出「訓練到 peak LR 附近才發散」
這種問題。

In [ ]:
from llmcls.train import train_fold

smoke = train_fold(
    fold=0, epochs=1, batch_size=8, max_len=512, lr=2e-5,
    lr_scheduler_type="constant_with_warmup", fp16=True,
    max_train_rows=3000, max_valid_rows=800, max_steps=400, eval_steps=200,
    output_dir="/kaggle/working/model/_smoke",
)
print(f"log loss={smoke['score']:.5f}  n_nonfinite={smoke['n_nonfinite']}  diverged={smoke['diverged']}")
assert not smoke["diverged"] and smoke["n_nonfinite"] == 0, (
    "加速設定（fp16=True）在煙霧測試就不穩定，先不要跑完整訓練"
)
print("煙霧測試通過，加速設定沒有引入不穩定")

## Label smoothing 實驗（milestone 3，先只在 fold 0 驗證，不直接動全部 5 folds）

白話說：訓練時原本要求模型對正確答案喊到 100% 確定，`label_smoothing=0.1` 把這個
目標調鬆成 90%、剩下 10% 分給另外兩類。log loss 對「很有信心但答錯」罰得特別重，
放鬆信心通常直接有幫助。

這個改動要重新訓練（不像 TTA/校準/ensemble 只改推論端就能後補），所以先只在
fold 0 花一次訓練時間（約 1.7 小時）驗證有沒有用，output_dir 跟主要的 5-fold
訓練分開，不會互相干擾，也不會被下面的「跳過已存在權重」邏輯誤判成同一份。
確認有用才決定要不要把全部 5 folds 重練一次套上這個設定。

比較基準：fold 0 目前（無 label smoothing）的 valid log loss 是 1.06840（下面
5-fold 訓練表格那一列）。TTA+校準那次實驗量到的 fold 間標準差是 0.00288——下面
這個分數至少要比 1.06840 好超過這個雜訊量級，才算是真的有幫助，不是雜訊。

In [ ]:
FOLD0_BASELINE_NO_LABEL_SMOOTHING = 1.06840

ls_result = train_fold(
    fold=0, epochs=2, batch_size=8, max_len=512, lr=2e-5,
    fp16=True, eval_steps=1500, eval_subset_rows=2000,
    label_smoothing=0.1,
    output_dir="/kaggle/working/model/_label_smoothing_fold0",
)
print(f"fold 0（label_smoothing=0.1）valid log loss: {ls_result['score']:.5f}")
print(f"fold 0 基準（無 label smoothing）: {FOLD0_BASELINE_NO_LABEL_SMOOTHING:.5f}")
print(f"差異：{FOLD0_BASELINE_NO_LABEL_SMOOTHING - ls_result['score']:+.5f}（正值代表 label smoothing 有幫助）")
assert not ls_result["diverged"] and ls_result["n_nonfinite"] == 0, (
    "label smoothing 訓練發散或有非有限值，不要拿這個結果做決定"
)

**決定要不要繼續**：如果上面的差異明顯大於 0.003（TTA/校準實驗量到的雜訊量級），
值得把全部 5 folds 重新訓練一次、套上 `label_smoothing=0.1`（把下面主要訓練迴圈
的 `train_fold(...)` 呼叫加上這個參數即可）。如果差異在雜訊範圍內或更差，放棄這條
路，去做 milestone 3 剩下的其他項目（訓練時 a/b 對調增強、`winner_tie` 特殊處理）。

煙霧測試過關後才跑完整訓練。valid log loss 必須小於 1.09861（ln 3）—— 這是本專案
判斷分數的唯一標準，也是 scripts/baseline_prior.py 在真實資料上印出的基準
（1.09723）。步進式存檔（每 1500 步）跟發散偵測保護都還在，不會再白燒一整個
epoch。`eval_subset_rows=2000` 讓訓練中途的評估變快，最終回報的分數保證是對
完整驗證集算出來的。

**這裡練滿全部 5 個 fold**——milestone 3 的 TTA + temperature scaling 只在 fold 0
單一份驗證集上量過改善（+0.00383），單一切分的量測有可能只是那份驗證集剛好對這個
手法有利，5 個 fold 各自獨立驗證同一個改善才有說服力。

**實測踩過的坑（已修復）**：第一次跑 5 folds 時，`_training_args()` 的
`output_dir` 跟 `trainer.save_model()` 的最終存檔目錄是同一個路徑，Trainer 自己
`save_steps` 週期性存的 `checkpoint-*/`（含 optimizer/scheduler state，體積是
模型本身的 2-3 倍）一直沒清掉，5 個 fold 疊起來直接把 Kaggle 磁碟塞爆
（`OSError: No space left on device`，練到 fold 4 過半才炸）。`train_fold()`
現在會在 `trainer.save_model()` 之後自動清掉 `output_dir` 底下的 `checkpoint-*/`，
每個 fold 只留最終權重（~700MB，不是 ~2.8GB）。

那次事故裡 fold 0-3 其實都順利練完、分數都贏過基準（1.06840 / 1.05461 / 1.07391 /
1.08028），只有 fold 4 沒存到——下面會先檢查 `/kaggle/input` 有沒有掛之前留下來的
權重，有的話直接複製過來、略過重新訓練，不用 5 個全部重練一次。

In [ ]:
import pathlib
import shutil
import time

from llmcls.config import MODEL_DIR, N_FOLDS

# 如果有掛之前的訓練成果（例如上一次中途出錯，這次接續跑），先複製過來、跳過
# 已經練好的 fold，不用整個重來。找不到就當作全新開始，不影響正常流程。
#
# 檔名是攤平的 `fold{N}__檔名`（不是巢狀資料夾）——上傳 Kaggle Dataset 時，
# 巢狀資料夾要嘛跳過、要嘛整個壓成一個 zip/tar，`--dir-mode` 實際行為（會不會
# 自動解壓縮回資料夾）沒把握，攤平成單層檔名最保險，不用賭 Kaggle 那端怎麼處理。
for _p in pathlib.Path("/kaggle/input").glob("**/fold*__model.safetensors"):
    fold_name, _ = _p.name.split("__", 1)
    dst = MODEL_DIR / fold_name
    if not dst.exists():
        dst.mkdir(parents=True)
        for _f in _p.parent.glob(f"{fold_name}__*"):
            _, filename = _f.name.split("__", 1)
            shutil.copy(_f, dst / filename)
        print(f"{fold_name} 已從先前的權重複製過來，略過重新訓練")

In [ ]:
fold_scores = {}
for fold in range(N_FOLDS):
    dst = MODEL_DIR / f"fold{fold}"
    if (dst / "model.safetensors").exists():
        print(f"fold {fold} 已經有存好的權重（{dst}），略過重新訓練")
        continue
    _t0 = time.time()
    result = train_fold(
        fold=fold, epochs=2, batch_size=8, max_len=512, lr=2e-5,
        fp16=True, eval_steps=1500, eval_subset_rows=2000,
    )
    print(f"fold {fold} 訓練總耗時：{time.time() - _t0:.0f}s")
    assert not result["diverged"] and result["n_nonfinite"] == 0, (
        f"fold {fold} 完整訓練發散或有非有限值，模型權重不可信，不要拿去推論"
    )
    fold_scores[fold] = result["score"]
    print(f"fold {fold} valid log loss: {result['score']:.5f}")

每個 fold 都已經存到 `MODEL_DIR/fold{N}`（預設 `/kaggle/working/model/fold{N}`）。

確認全部贏過基準之後：**Save Version**（Save & Run All），存出的 Version 的
`/kaggle/working/model/` 就會變成一個新的 Kaggle Dataset（在 Notebook 的 Output
分頁），下一步把它掛進 `calibrate_folds.py`（重新驗證 TTA/校準在 5 個 fold 上是否
一致改善）跟 `infer_deberta.py`。

In [ ]:
print("五個 fold 的 valid log loss：")
for fold, score in fold_scores.items():
    print(f"  fold {fold}: {score:.5f}")